In [2]:
!pip install datasets pandas nltk spacy scikit-learn sqlite-utils
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [21]:
import pandas as pd
import spacy
import nltk
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sqlite3

nltk.download('wordnet')
from nltk.corpus import wordnet

# Initialize a global SQLite in-memory database connection
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
!wget https://github.com/salesforce/WikiSQL/raw/master/data.tar.bz2
!tar -xvjf data.tar.bz2

--2026-03-10 00:57:05--  https://github.com/salesforce/WikiSQL/raw/master/data.tar.bz2
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/salesforce/WikiSQL/master/data.tar.bz2 [following]
--2026-03-10 00:57:06--  https://raw.githubusercontent.com/salesforce/WikiSQL/master/data.tar.bz2
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26164664 (25M) [application/octet-stream]
Saving to: ‘data.tar.bz2’

data.tar.bz2        100%[===================>]  24.95M   140MB/s    in 0.2s    

2026-03-10 00:57:09 (140 MB/s) - ‘data.tar.bz2’ saved [26164664/26164664]

data/
data/train.jsonl
data/test.tables.j

In [4]:
import pandas as pd
import json

data = []

with open("data/train.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

print(data[1])

{'phase': 1, 'table_id': '1-1000181-1', 'question': 'What is the current series where the new series began in June 2011?', 'sql': {'sel': 4, 'conds': [[5, 0, 'New series began in June 2011']], 'agg': 0}}


In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")


##Table Word Extraction

In [6]:
def extract_keywords(question):

    doc = nlp(question)
    keywords = []

    for token in doc:
        if token.pos_ in ["NOUN","PROPN"]:
            keywords.append(token.text.lower())

    return keywords


question = data[1]["question"]

print("Question:", question)
print("Keywords:", extract_keywords(question))

Question: What is the current series where the new series began in June 2011?
Keywords: ['series', 'series', 'june']


##Synonym Processing

In [7]:
import nltk
nltk.download('wordnet')

from nltk.corpus import wordnet

def get_synonyms(word):

    synonyms = []

    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.append(lemma.name().lower())

    return list(set(synonyms))

keywords = extract_keywords(question)
for word in keywords:
    print(word, "->", get_synonyms(word))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


series -> ['serial', 'series', 'serial_publication']
series -> ['serial', 'series', 'serial_publication']
june -> ['june']


##Table Selection (Similarity)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def column_similarity(question, columns):

    texts = [question] + columns
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform(texts)
    similarity = cosine_similarity(vectors[0:1], vectors[1:])
    best_index = similarity.argmax()
    best_column = columns[best_index]
    return best_column, similarity

##SQL Query Generation

In [9]:
def generate_sql(column, table, condition_column, value):

    sql = f"SELECT {column} FROM {table} WHERE {condition_column} = '{value}'"

    return sql


query = generate_sql(
    column="Current Series",
    table="SeriesTable",
    condition_column="New Series Began",
    value="June 2011"
)

print(query)

SELECT Current Series FROM SeriesTable WHERE New Series Began = 'June 2011'


##Execute SQL Query

In [22]:
# Removed: import sqlite3, conn = sqlite3.connect(":memory:"), cursor = conn.cursor()

cursor.execute("""
CREATE TABLE SeriesTable (
Current_Series TEXT,
New_Series_Began TEXT,
Network TEXT
)
""")

cursor.execute("""
INSERT INTO SeriesTable VALUES
('Series 5','June 2011','BBC')
""")

cursor.execute("""
CREATE TABLE Players (
Player TEXT,
No INTEGER,
Nationality TEXT,
Position TEXT
);
""")

cursor.execute("""
INSERT INTO Players VALUES
('Loren Woods', 12, 'USA', 'Center'),
('Michael Jordan', 23, 'USA', 'Shooting Guard'),
('LeBron James', 6, 'USA', 'Small Forward');
""")

# This part was for testing the SeriesTable and can remain or be moved if desired
query = """
SELECT Current_Series
FROM SeriesTable
WHERE New_Series_Began = 'June 2011'
"""

cursor.execute(query)

print(cursor.fetchall())

[('Series 5',)]


In [25]:
columns = ['Player','No','Nationality','Position']

table_name = "Players"

question = input("Ask a question: ")

keywords = extract_keywords(question)

best_column, _ = column_similarity(question, columns)

value = extract_value(question)

condition_column = detect_condition_column(columns, value, question)

query = generate_sql(best_column, table_name, condition_column, value)

cursor.execute(query)

result = cursor.fetchall()

print("Generated SQL:", query)
print("Answer:", result)

Ask a question: nationality is Loren Woods
Generated SQL: SELECT Nationality FROM Players WHERE Player = 'Loren Woods'
Answer: [('USA',)]


In [15]:
def extract_value(question):
    doc = nlp(question)
    value = []
    # Heuristically look for named entities as potential values
    for ent in doc.ents:
        # Consider PERSON, DATE, GPE (geopolitical entity), ORG (organization) etc.
        # This can be refined based on specific use cases
        if ent.label_ in ["PERSON", "DATE", "GPE", "ORG", "LOC", "NORP"]:
            value.append(ent.text)
    # If no specific entities, try to extract something that looks like a value
    # This part might need further refinement based on question patterns
    if not value:
        # A very simple heuristic: often the value is a proper noun or number not covered by NER
        # For example, in "What position does Loren Woods play?", "Loren Woods" is a PERSON.
        # In "New series began in June 2011", "June 2011" is a DATE.
        # If NER misses, try to find proper nouns that might be the value
        keywords_from_question = extract_keywords(question)
        for token in doc:
            if token.pos_ == "PROPN" and token.text.lower() not in keywords_from_question:
                value.append(token.text)

    # For simplicity, return the first identified value or an empty string if none found
    return " ".join(value) if value else ""

def detect_condition_column(columns, extracted_value, question):
    # This is a simplified approach. In a real system, this would be more sophisticated.
    # One way is to check which column name is most similar to the extracted value
    # or to the context around the extracted value in the question.

    # If an extracted_value is present, try to match it against column names
    if extracted_value:
        # Use column_similarity to find the best matching column for the extracted value
        # Treat the extracted value as a 'mini-question' to find the relevant column
        best_col_for_value, _ = column_similarity(extracted_value, columns)
        return best_col_for_value

    # If no specific value is extracted, or if the above doesn't yield a good result,
    # a more advanced approach would be needed. For now, we might default or raise an error.
    return "" # Or handle this case more robustly
